In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, when, sum

In [2]:
spark = SparkSession.builder \
    .appName("SmartHomeEnergyTracker") \
    .getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()

Saving sensor_logs.csv to sensor_logs.csv


In [4]:
df = spark.read.csv(
    "sensor_logs.csv",
    header=True,
    inferSchema=True
)

df.show()

+---------+-----------+----------+-------------------+
|device_id|device_name|energy_kwh|          timestamp|
+---------+-----------+----------+-------------------+
|        1|        Fan|       2.5|2026-06-15 08:30:00|
|        2|         TV|       1.8|2026-06-15 20:15:00|
|        3|     Fridge|       3.2|2026-06-15 14:00:00|
|        4|         AC|       5.6|2026-06-15 23:00:00|
|        5|      Light|       0.7|2026-06-15 19:30:00|
|        6|        Fan|       1.9|2026-06-15 06:45:00|
|        7|         AC|       4.8|2026-06-15 21:15:00|
+---------+-----------+----------+-------------------+



In [5]:
df = df.withColumn(
    "hour",
    hour(col("timestamp"))
)

df.show()

+---------+-----------+----------+-------------------+----+
|device_id|device_name|energy_kwh|          timestamp|hour|
+---------+-----------+----------+-------------------+----+
|        1|        Fan|       2.5|2026-06-15 08:30:00|   8|
|        2|         TV|       1.8|2026-06-15 20:15:00|  20|
|        3|     Fridge|       3.2|2026-06-15 14:00:00|  14|
|        4|         AC|       5.6|2026-06-15 23:00:00|  23|
|        5|      Light|       0.7|2026-06-15 19:30:00|  19|
|        6|        Fan|       1.9|2026-06-15 06:45:00|   6|
|        7|         AC|       4.8|2026-06-15 21:15:00|  21|
+---------+-----------+----------+-------------------+----+



In [6]:
df = df.withColumn(
    "usage_type",
    when(
        (col("hour") >= 6) & (col("hour") < 22),
        "Peak"
    ).otherwise("Off-Peak")
)

df.show()

+---------+-----------+----------+-------------------+----+----------+
|device_id|device_name|energy_kwh|          timestamp|hour|usage_type|
+---------+-----------+----------+-------------------+----+----------+
|        1|        Fan|       2.5|2026-06-15 08:30:00|   8|      Peak|
|        2|         TV|       1.8|2026-06-15 20:15:00|  20|      Peak|
|        3|     Fridge|       3.2|2026-06-15 14:00:00|  14|      Peak|
|        4|         AC|       5.6|2026-06-15 23:00:00|  23|  Off-Peak|
|        5|      Light|       0.7|2026-06-15 19:30:00|  19|      Peak|
|        6|        Fan|       1.9|2026-06-15 06:45:00|   6|      Peak|
|        7|         AC|       4.8|2026-06-15 21:15:00|  21|      Peak|
+---------+-----------+----------+-------------------+----+----------+



In [7]:
device_usage = df.groupBy(
    "device_name",
    "usage_type"
).agg(
    sum("energy_kwh").alias("total_energy")
)

device_usage.show()

+-----------+----------+------------+
|device_name|usage_type|total_energy|
+-----------+----------+------------+
|         AC|      Peak|         4.8|
|        Fan|      Peak|         4.4|
|         AC|  Off-Peak|         5.6|
|         TV|      Peak|         1.8|
|     Fridge|      Peak|         3.2|
|      Light|      Peak|         0.7|
+-----------+----------+------------+



In [8]:
top_devices = df.groupBy(
    "device_name"
).agg(
    sum("energy_kwh").alias("total_energy")
).orderBy(
    col("total_energy").desc()
)

top_devices.show()

+-----------+------------------+
|device_name|      total_energy|
+-----------+------------------+
|         AC|10.399999999999999|
|        Fan|               4.4|
|     Fridge|               3.2|
|         TV|               1.8|
|      Light|               0.7|
+-----------+------------------+



In [9]:
top_devices = df.groupBy(
    "device_name"
).agg(
    sum("energy_kwh").alias("total_energy")
).orderBy(
    col("total_energy").desc()
)

top_devices.show()

+-----------+------------------+
|device_name|      total_energy|
+-----------+------------------+
|         AC|10.399999999999999|
|        Fan|               4.4|
|     Fridge|               3.2|
|         TV|               1.8|
|      Light|               0.7|
+-----------+------------------+



In [10]:
top_devices.toPandas().to_csv(
    "top_devices_output.csv",
    index=False
)